In [2]:
#Imports 
import pandas as pd
import re
import numpy as np
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
BRONZE_PATH = "../data/bronze/"
SILVER_PATH = "../data/silver/"



In [3]:
# Load all bronze files
print("Loading bronze files")
cert_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "certifications_bronze.csv"))
clubs_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "clubs_bronze.csv"))
results_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "results_bronze.csv"))

print("\n" + "="*60)
print("BRONZE FILE SHAPES (Before cleaning)")
print("="*60)
print(f"Certifications: {cert_bronze.shape} rows, {cert_bronze.shape[1]} columns")
print(f"Clubs: {clubs_bronze.shape} rows, {clubs_bronze.shape[1]} columns")
print(f"Results: {results_bronze.shape} rows, {results_bronze.shape[1]} columns")

Loading bronze files

BRONZE FILE SHAPES (Before cleaning)
Certifications: (21001, 12) rows, 12 columns
Clubs: (436, 19) rows, 19 columns
Results: (112375, 14) rows, 14 columns


In [4]:
#Checking null values
print("="*60)
print("NULL VALUES IN BRONZE")
print("="*60)

print("\n1. Certifications null counts:")
null_cert = cert_bronze.isnull().sum()
null_cert = null_cert[null_cert > 0]
print(null_cert if len(null_cert) > 0 else "  No null values found")

print("\n2. Clubs null counts:")
null_clubs = clubs_bronze.isnull().sum()
null_clubs = null_clubs[null_clubs > 0]
print(null_clubs if len(null_clubs) > 0 else "  No null values found")

print("\n3. Results null counts:")
null_results = results_bronze.isnull().sum()
null_results = null_results[null_results > 0]
print(null_results if len(null_results) > 0 else "  No null values found")

NULL VALUES IN BRONZE

1. Certifications null counts:
Club                                            780
Code                                            780
Person type                                     780
Gender                                          780
DOB                                            1990
Age                                             780
Mental Handicap (SOB has this certificate)    11188
Parents Consent (SOB has this certificate)    14358
HAP (SOB has this certificate)                15474
Unified Partner (SOB has this certificate)    20818
dtype: int64

2. Clubs null counts:
Address (Street and Number)      2
Zipcode                          3
Province                         6
Country                         74
Participation Games 2015       157
Participation Games 2022       174
Participation Games 2023       136
Participation Games 2024       136
Participation Games 2025       116
dtype: int64

3. Results null counts:
Code                79
Club          

In [5]:
#Cleaning "Certifications"
print("="*60)
print("SILVER: Cleaning Certifications")
print("="*60)

df = cert_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Standardize Person type 
df['person_type_clean'] = df['Person type'].astype(str).str.lower().str.strip()
print(f"  Person type unique values: {df['person_type_clean'].unique()}")

# Step 2: Standardize Gender (M/F)
df['gender_clean'] = df['Gender'].astype(str).str.upper()
print(f"  Gender unique values: {df['gender_clean'].unique()}")

# Step 3: Convert boolean columns to 0/1 and fill missing with 0
bool_cols = [
    'Mental Handicap (SOB has this certificate)',
    'Parents Consent (SOB has this certificate)',
    'HAP (SOB has this certificate)',
    'Unified Partner (SOB has this certificate)'
]

for col in bool_cols:
    # Convert True/False to 1/0, fill NaN with 0
    df[col] = df[col].fillna(0).astype(int)
    # Also handle string True/False if present
    df[col] = df[col].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)
print(f"  Boolean columns converted to 0/1")

# Step 4: Flag missing DOB
df['dob_missing'] = df['DOB'].isnull().astype(int)
print(f"  DOB missing: {df['dob_missing'].sum()} rows ({df['dob_missing'].sum()/len(df)*100:.1f}%)")

# Step 5: Remove rows with null Person type
before = len(df)
df = df[df['person_type_clean'].notna() & (df['person_type_clean'] != 'nan')]
after = len(df)
print(f"  Removed {before - after} rows with missing Person type")

# Step 6: Check null values after cleaning
print(f"\nFinal shape: {df.shape}")
print(f"Null counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No null values remaining")

# Save to silver
output_path = os.path.join(SILVER_PATH, "certifications_silver.csv")
df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Certifications
Starting shape: (21001, 12)
  Person type unique values: <ArrowStringArray>
[        'athlete',           'coach', 'unified partner',               nan,
       'volunteer',             'vip',           'staff',         'medical',
      'head coach',         'manager',        'security',   'family member',
           'a-hod',           'media',        'as-staff']
Length: 15, dtype: str
  Gender unique values: <ArrowStringArray>
['M', 'F', nan, 'U']
Length: 4, dtype: str
  Boolean columns converted to 0/1
  DOB missing: 1990 rows (9.5%)
  Removed 780 rows with missing Person type

Final shape: (20221, 15)
Null counts after cleaning:
DOB    1210
dtype: int64

Saved to: ../data/silver/certifications_silver.csv


In [6]:
#Validation certifications

df_test = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
print("Certifications validation:")
print(f"  Shape: {df_test.shape}")
print(f"  Columns: {df_test.columns.tolist()}")
print(f"  Sample person_type values: {df_test['person_type_clean'].head(3).tolist()}")
print(f"  Boolean column sample (HAP): {df_test['HAP (SOB has this certificate)'].head(3).tolist()}")

Certifications validation:
  Shape: (20221, 15)
  Columns: ['Club', 'Code', 'Person type', 'Gender', 'DOB', 'Age', 'Mental Handicap (SOB has this certificate)', 'Parents Consent (SOB has this certificate)', 'HAP (SOB has this certificate)', 'Unified Partner (SOB has this certificate)', 'bronze_timestamp', 'bronze_source', 'person_type_clean', 'gender_clean', 'dob_missing']
  Sample person_type values: ['athlete', 'coach', 'athlete']
  Boolean column sample (HAP): [1, 0, 0]
